# Human in the Loop

- Sometimes it is very useful to keep a human in the loop to see what an agent is doing
- This is easy to do with LangGraph
- Interrupt before action should be applied using the tools called by the AI so that the huamn can decide whether the agent should continue the action or not

#### _Create agent with huamn in the loop:_

Import from langgraph and langchain:

    from langgraph.graph import StateGraph, END
    from typing import TypedDict, Annotated
    import operator
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
    from langchain_openai import ChatOpenAI
    from langchain_community.tools.tavily_search import TavilySearchResults   

Import Sqlite saver checkpointer and use it as memory:

    from langgraph.checkpoint.sqlite import SqliteSaver

    memory = SqliteSaver.from_conn_string(":memory:")

=> Previously for the agent state, we annotated the messages list with the operator add to add messages into the existing messages array

=> For human in the loop, we may want to replace existing messages using a reduce messages function

Reduce Messages function:

=> This function looks for msgs with same ID and if notices that someone is inserting another message with same ID as one that exists, it will be replaced. Otherwise, it will be appended.

=> Instead of the operator add to add messages into the messages dictionary, the message will be replaces if the same id message exists or it will be appended.

    from uuid import uuid4
    from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

    def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
        # assign ids to messages that don't have them
        for message in right:
            if not message.id:
                message.id = str(uuid4())
        # merge the new messages with the existing messages
        merged = left.copy()
        for message in right:
            for i, existing in enumerate(merged):
                # replace any existing messages with the same id
                if existing.id == message.id:
                    merged[i] = message
                    break
            else:
                # append any new messages to the end
                merged.append(message)
        return merged


Create Agent state of typed dict with anottated messages and reduce messages function:

    class AgentState(TypedDict):
        messages: Annotated[list[AnyMessage], reduce_messages]

Create tavily search result tool:

    tool = TavilySearchResults(max_results=2)

Create Agent using LangGraph with manual human approval:

=> Add Interrupt before action i.e before a tool call is applied to graph compile along with checkpointer

=> This permits manual approval before we run any tools

=> Useful when you want tool to be executed correctly

    class Agent:
        def __init__(self, model, tools, system="", checkpointer=None):
            self.system = system
            graph = StateGraph(AgentState)
            graph.add_node("llm", self.call_openai)
            graph.add_node("action", self.take_action)
            graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
            graph.add_edge("action", "llm")
            graph.set_entry_point("llm")
            self.graph = graph.compile(
                checkpointer=checkpointer,
                interrupt_before=["action"]
            )
            self.tools = {t.name: t for t in tools}
            self.model = model.bind_tools(tools)

        def call_openai(self, state: AgentState):
            messages = state['messages']
            if self.system:
                messages = [SystemMessage(content=self.system)] + messages
            message = self.model.invoke(messages)
            return {'messages': [message]}

        def exists_action(self, state: AgentState):
            print(state)
            result = state['messages'][-1]
            return len(result.tool_calls) > 0

        def take_action(self, state: AgentState):
            tool_calls = state['messages'][-1].tool_calls
            results = []
            for t in tool_calls:
                print(f"Calling: {t}")
                result = self.tools[t['name']].invoke(t['args'])
                results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            print("Back to the model!")
            return {'messages': results}

Initialise agent with model, tool, system prompt and checkpointer:

    prompt = """You are a smart research assistant. Use the search engine to look up information. \
    You are allowed to make multiple calls (either together or in sequence). \
    Only look up information when you are sure of what you want. \
    If you need to look up some information before asking a follow up question, you are allowed to do that!
    """
    model = ChatOpenAI(model="gpt-3.5-turbo")
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

Create human message and cofig thread with id:

    messages = [HumanMessage(content="Whats the weather in SF?")]
    thread = {"configurable": {"thread_id": "1"}}


Stream agent event with messages and thread:

    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

=> It returns the stream of events within the agent but stops after the AI message as the AI msg asks to call a tool and we have an interrupt before action.

Get current state of the graph for this thread:

    abot.graph.get_state(thread)

Can get next state in the graph for this thread which is the action node:

    abot.graph.get_state(thread).next

To continue same stream, use same thread config and pass None as the input:

    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

=> We get the tool message for calling the tool.

=> We also get the final AI message with the answer to the query.

#### _Loop through agent to ask whether to proceed action or not:_

=> While next state is thread exists i.e has to perform an action, the loop will ask whether to proceed action or not

=> If you input 'y' or 'yes', the agent stream will continue and call the tools to perform the action

=> If you input 'no' to not proceed, the while loop will break and not proceed to call action

    messages = [HumanMessage("Whats the weather in LA?")]
    thread = {"configurable": {"thread_id": "2"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)
    while abot.graph.get_state(thread).next:
        print("\n", abot.graph.get_state(thread),"\n")
        _input = input("proceed?")
        if _input != "y":
            print("aborting")
            break
        for event in abot.graph.stream(None, thread):
            for v in event.values():
                print(v)

#### _State Memory:_

- As graph is executing, a snapshot of each state is taken and stored in memory
- Snapshot contains agent state and other useful things like thread and thread ts (unique identifier for each snapshot)
- Thread ts can be used to access snapshots
- Can get state with thread 
- Can get state history which returns an iterator of all the state snapshots
- Can use the iterator to get access to all the unique identifiers for each state
- Can update state and store back into memory. The agent will run the updated state if you continue the stream 

Ask query to agent:

    messages = [HumanMessage("Whats the weather in LA?")]
    thread = {"configurable": {"thread_id": "3"}}
    for event in abot.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

Get state of the graph:

    abot.graph.get_state(thread)

EXAMPLE: To modify or update the query to get weather from another location instead:

Save current state of the graph to a variable:

    current_values = abot.graph.get_state(thread)

Get last message in the state:

    current_values.values['messages'][-1]

Get list of tool calls mentioned in the last message:

    current_values.values['messages'][-1].tool_calls

Update tool calls and query with tool call id:

    _id = current_values.values['messages'][-1].tool_calls[0]['id']
    current_values.values['messages'][-1].tool_calls = [
        {'name': 'tavily_search_results_json',
    'args': {'query': 'current weather in Louisiana'},
    'id': _id}
    ]

Update state in graph with thread and current values variable:

    abot.graph.update_state(thread, current_values.values)

Get current state of the graph:

    abot.graph.get_state(thread)

=> The query is updated

Continue agent stread with none as input and thread:

    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

Visit previous state of the agent using get state history:

    states = []
    for state in abot.graph.get_state_history(thread):
        print(state)
        print('--')
        states.append(state)

Resume agent from the checkpoint where it was search for Los angeles by configuring with that message to reume's config in the agent graph:

    to_replay = states[-3]

    to_replay

    for event in abot.graph.stream(None, to_replay.config):
        for k, v in event.items():
            print(v)

=> Gets response back from tavily based on the resumed message query

=> Gets final answer from the AI message

Go back to previoud state to edit:

    to_replay   

Update state in the graph to get weather from accuweather:

    _id = to_replay.values['messages'][-1].tool_calls[0]['id']
    to_replay.values['messages'][-1].tool_calls = [{'name': 'tavily_search_results_json',
    'args': {'query': 'current weather in LA, accuweather'},
    'id': _id}]

    branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

    for event in abot.graph.stream(None, branch_state):
        for k, v in event.items():
            if k != "__end__":
                print(v)

<u>EXAMPLE: Add mock response from tool call to state by appending to messages</u>

Add message to state at a given time:

    to_replay

Get id of tool call that was supposed to be made:

    _id = to_replay.values['messages'][-1].tool_calls[0]['id']

Create state update with a mock response from tavily tool without calling tavily:

=> Annoted message as tool message and Use tool call id, tool name, content as mock response

    state_update = {"messages": [ToolMessage(
        tool_call_id=_id,
        name="tavily_search_results_json",
        content="54 degree celcius",
    )]}

Update state of the graph and make the update as action node:

    branch_and_add = abot.graph.update_state(
        to_replay.config, 
        state_update, 
        as_node="action")

=> So it doesn't go to action node again and call tavily tool but uses the mock response made 

=> We act as if we were the action node responce after calling tool

Continue agent stream with none input and updated state:

    for event in abot.graph.stream(None, branch_and_add):
        for k, v in event.items():
            print(v)